# Loading an Earth Engine Vector Ecosystem Map

This notebook demonstrates how to load an Earth Engine FeatureCollection as an ecosystem map using the `iucn-get-data` package, and display it using Earth Engine tile rendering via lonboard's `BitmapTileLayer`.

This avoids downloading geometry to the client (as `load_parquet.ipynb` does with `PolygonLayer`), making it suitable for large datasets.

In [ ]:
import os

import ee
from google.auth import default
from lonboard import Map, BitmapTileLayer
from lonboard.view_state import MapViewState

import iucn_get_data
from iucn_get_data import open_ecosystem_map, Typology

## Initialize Earth Engine

In [ ]:
credentials, _ = default(scopes=[
    'https://www.googleapis.com/auth/earthengine',
    'https://www.googleapis.com/auth/cloud-platform',
])
ee.Initialize(credentials=credentials, project=os.environ['GOOGLE_CLOUD_PROJECT'])

## Load an Earth Engine FeatureCollection

Use `open_ecosystem_map()` to load an Earth Engine vector asset. The EE backend is auto-detected from the asset ID. Specify the column names that map to GET Level 3 (Ecosystem Functional Group) and Level 4/5/6 (ecosystem type) codes.

In [ ]:
asset_id = 'projects/goog-rle-assessments/assets/colombia/GETCol'

ecosystems = open_ecosystem_map(
    asset_id,
    get_level3_column='EFG1',
    get_level456_column='Glob_Typol',
)
ecosystems

## Display on an interactive map using Earth Engine tiles

Instead of downloading geometry to render client-side with `PolygonLayer`, we paint the `ee.FeatureCollection` into an `ee.Image` and use `getMapId()` to generate tile URLs. The tiles are displayed via lonboard's `BitmapTileLayer`.

In [ ]:
colombia_view_state = MapViewState(longitude=-70.6, latitude=6.0, zoom=4)
m_ecosystems = ecosystems.to_map(view_state=colombia_view_state)
m_ecosystems

## Colored by Functional Group (GET Level 3)

Paint each feature with a unique value per functional group, then apply a palette to distinguish them.

In [ ]:
m_fg = ecosystems.to_functional_group_map(view_state=colombia_view_state)
m_fg

In [ ]:
from ipywidgets import jslink

# Use .map to access the underlying lonboard.Map from ClickableMap
jslink((m_ecosystems.map, 'view_state'), (m_fg.map, 'view_state'))

In [ ]:
ecosystems.to_biome_map(view_state=colombia_view_state)

In [ ]:
ecosystems.to_realm_map(view_state=colombia_view_state)

## Extract the functional group DataFrame

`functional_group_dataframe()` returns distinct ecosystem types indexed by GET Level 3 and Level 4/5/6 codes, computed server-side via Earth Engine.

In [ ]:
fg_df = ecosystems.functional_group_dataframe()
fg_df

## Combine with the IUCN GET Typology

Pass the functional group DataFrame into `Typology` to see how the ecosystems fit into the full IUCN Global Ecosystem Typology hierarchy.

In [ ]:
typology = Typology(
    ecosystems=fg_df.reset_index(),
    ecosystems_functional_group_column='EFG1',
)
typology